# CNN Feature Visualization, Parameter Audit & Critical Analysis
### Deep Learning & Computer Vision — Practical Laboratory Assignment

| | |
|---|---|
| **Student Name** | Muhammad Asad Mashood |
| **Registration No.** | FA23-BAI-039 |
| **Course / Section** | _fill in_ |
| **Submission Date** | _fill in_ |

---

**How this notebook is built (what's different about it)**

1. **No black-box convolution.** Conv, ReLU, Max/Avg pooling are written from scratch in NumPy (exactly the cross-correlation a CNN layer computes), and then **verified to 1e-12** against SciPy and PyTorch. So every pixel in the grid can be explained by hand in the viva.
2. **Two input images.** A real image (brick wall: dense horizontal + vertical edges) **and** a synthetic *probe image* with a known geometry (rectangle, disc, smooth ramp). On the probe the correct answer is known in advance, so the feature maps can be checked against theory.
3. **Every written answer is backed by a measurement.** Each Part III answer quotes numbers printed by the code above it (percent of pixels blacked out, energy kept, edge survival under pooling, etc.), not general statements.
4. **Each claim is tested with an experiment:** activation swap (ReLU / Leaky / Sigmoid), a dark-text-on-white pooling test, numerical proof of ReLU–MaxPool commutativity plus a timing benchmark, empirical receptive-field measurement with gradients, and a real training run that **reproduces filter collapse and then fixes it**.
5. **A finding that changes the textbook answer to Q3:** replacing a 5×5 layer with two 3×3 layers does **not** always save parameters. It depends on the width of the middle layer, and in Model B it can cost *more*. Shown with numbers and verified in Keras.

**Runtime:** Google Colab (CPU is enough; GPU only speeds up Part III-Q5). TensorFlow/Keras and PyTorch are both pre-installed in Colab.

### Contents
0. Setup
1. **Part I** — Input images · five 3×3 kernels · from-scratch ops + verification · pipeline · **5×4 grid** · quantitative evidence
2. **Part II** — Hand calculation · Keras `model.summary()` · PyTorch cross-check · audit table · receptive field · compute cost
3. **Part III** — Experiments + answers for Q1–Q5
4. Viva cheat-sheet · export figures

## 0 · Setup

In [ ]:
import os, time, timeit, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from scipy.signal import correlate2d
from skimage import data, color, transform, io
import cv2

warnings.filterwarnings('ignore')
np.random.seed(42)
pd.set_option('display.precision', 3)
pd.set_option('display.width', 200)

FIG_DIR = 'figures'
os.makedirs(FIG_DIR, exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 200, 'font.size': 9,
                     'axes.titlesize': 9, 'axes.labelsize': 9})

def savefig(fig, name):
    path = f'{FIG_DIR}/{name}.png'
    fig.savefig(path, bbox_inches='tight')
    print('saved ->', path)

print('NumPy', np.__version__, '| OpenCV', cv2.__version__)

# PART I — Filter Pipeline Implementation & Visualization

## 1.1 Input images

* **Main image:** `skimage.data.brick()`, a brick wall. The mortar gives bright **horizontal** lines and short bright **vertical** joints, which is the "crisp horizontal and vertical structural edges" the brief asks for. Set `USE_MY_OWN_IMAGE = True` to upload your own photo instead (e.g. a building).
* **Probe image (synthetic control):** a bright rectangle (2 horizontal + 2 vertical edges with *known* polarity), a disc (edges in every orientation) and a smooth ramp (a gradient with no edge). On this image every feature-map value can be predicted on paper first.

Images are converted to grayscale in **[0, 1]** and resized to 256×256.

In [ ]:
USE_MY_OWN_IMAGE = False     # True -> upload your own picture in Colab
IMG_SIZE = 256

def load_image():
    if USE_MY_OWN_IMAGE:
        from google.colab import files
        up = files.upload()
        path = list(up.keys())[0]
        img, src = io.imread(path), f'uploaded file: {path}'
    else:
        img, src = data.brick(), 'skimage.data.brick() (brick wall, 512x512 grayscale)'
    if img.ndim == 3:
        img = color.rgb2gray(img[..., :3])          # -> float in [0,1]
    img = img.astype(np.float64)
    if img.max() > 1.0:
        img = img / 255.0
    img = transform.resize(img, (IMG_SIZE, IMG_SIZE), anti_aliasing=True)
    return img, src

def make_probe(n=128):
    p = np.zeros((n, n))
    p[24:64, 24:104] = 1.0                                   # rectangle: top/bottom = horizontal edges, left/right = vertical
    yy, xx = np.mgrid[:n, :n]
    p[((yy - 96) ** 2 + (xx - 40) ** 2) < 18 ** 2] = 0.6     # disc: edges in every orientation
    p[80:112, 72:120] = np.linspace(0, 1, 48)[None, :]       # horizontal ramp: smooth gradient, no sharp edge
    return p

img, img_src = load_image()
probe = make_probe()
print('Main image :', img_src, '->', img.shape, f'range [{img.min():.2f}, {img.max():.2f}]')
print('Probe image:', probe.shape)

fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(img, cmap='gray', vmin=0, vmax=1); ax[0].set_title('Main input (grayscale, [0,1])')
ax[1].imshow(probe, cmap='gray', vmin=0, vmax=1); ax[1].set_title('Probe image (known geometry)')
ax[2].hist(img.ravel(), bins=60, color='0.3'); ax[2].set_title('Main image intensity histogram')
for a in ax[:2]: a.axis('off')
plt.tight_layout(); savefig(fig, '00_inputs'); plt.show()

## 1.2 The five 3×3 kernels

**A note on names, worth raising in the viva.** "Sobel-X" means the *derivative along x*. A derivative along x is large where intensity changes left→right, which happens on **vertical** edges. So *Sobel-X detects vertical edges* and *Sobel-Y / Prewitt-Y detects horizontal edges*. The brief's wording mixes the two naming conventions. In this notebook each kernel is named **by the edge orientation it lights up**, and the derivative direction is shown in brackets so there is no ambiguity.

| # | Kernel | Matrix | Sum (DC gain) |
|---|---|---|---|
| 1 | Horizontal edge — Prewitt, ∂/∂y | `[[-1,-1,-1],[0,0,0],[1,1,1]]` | 0 |
| 2 | Vertical edge — Sobel, ∂/∂x | `[[-1,0,1],[-2,0,2],[-1,0,1]]` | 0 |
| 3 | Sharpen | `[[0,-1,0],[-1,5,-1],[0,-1,0]]` | 1 |
| 4 | Gaussian blur | `[[1,2,1],[2,4,2],[1,2,1]] / 16` | 1 |
| 5 | Emboss | `[[-2,-1,0],[-1,1,1],[0,1,2]]` | 1 |

The **kernel sum** predicts what ReLU will do before anything runs. On a flat patch of brightness *v*, the output is *v × sum*.
* sum = 0 (edge kernels): flat regions → 0 and the output is centred on zero, so roughly half the response is negative and **ReLU will remove a large part of it**.
* sum = 1 (sharpen, blur, emboss): flat regions → *v* ≥ 0, so the output is mostly positive and **ReLU changes little**.

The second row of the figure shows each kernel's **frequency response** (|FFT|). Edge and sharpen kernels are high-pass, blur is low-pass. This explains the pooling results in Q2.

In [ ]:
KERNELS = {
    'Horizontal edge\n(Prewitt, d/dy)': np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]], float),
    'Vertical edge\n(Sobel, d/dx)':     np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float),
    'Sharpen':                          np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], float),
    'Gaussian blur':                    np.array([[1, 2, 1], [2, 4, 2], [1, 2, 1]], float) / 16,
    'Emboss':                           np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], float),
}
SHORT = dict(zip(KERNELS, ['H-edge', 'V-edge', 'Sharpen', 'Blur', 'Emboss']))

fig, axes = plt.subplots(2, 5, figsize=(16, 6.5))
for i, (name, k) in enumerate(KERNELS.items()):
    ax = axes[0, i]
    lim = np.abs(k).max()
    ax.imshow(k, cmap='RdBu_r', vmin=-lim, vmax=lim)
    for (r, c), v in np.ndenumerate(k):
        ax.text(c, r, f'{v:.3g}', ha='center', va='center', fontsize=11, fontweight='bold')
    ax.set_title(f'{name}\nsum = {k.sum():.2f}'); ax.set_xticks([]); ax.set_yticks([])
    F_ = np.abs(np.fft.fftshift(np.fft.fft2(k, s=(64, 64))))
    axes[1, i].imshow(F_, cmap='viridis'); axes[1, i].set_title('|FFT| (centre = low freq.)')
    axes[1, i].set_xticks([]); axes[1, i].set_yticks([])
plt.suptitle('Five 3×3 kernels (top) and their frequency responses (bottom)', fontsize=12)
plt.tight_layout(); savefig(fig, '01_kernels'); plt.show()

## 1.3 From-scratch operations + verification

A CNN "convolution" layer actually computes **cross-correlation** (the kernel is *not* flipped):

$$Y[i,j] = \sum_{u=-1}^{1}\sum_{v=-1}^{1} K[u,v]\; X[i+u,\, j+v]$$

This matters for direction-sensitive kernels. With the H-edge kernel (−1 row on top, +1 row at the bottom), the output is **(pixels below) − (pixels above)**. That sign is exactly what decides which pixels go black after ReLU.

**Padding choice:** *reflect* padding, so the output keeps the 256×256 size **without inventing a fake edge** at the image border. Zero padding would put a bright frame of edge responses around the image and pollute the statistics.

In [ ]:
def conv2d(x, k, pad='reflect'):
    """Single-channel CNN-style convolution (cross-correlation), stride 1, 'same' output size."""
    kh, kw = k.shape
    xp = np.pad(x, ((kh // 2, kh // 2), (kw // 2, kw // 2)), mode=pad)
    windows = sliding_window_view(xp, (kh, kw))          # (H, W, kh, kw) view, no copy
    return np.einsum('ijkl,kl->ij', windows, k)

def relu(x):               return np.maximum(0.0, x)
def leaky_relu(x, a=0.1):  return np.where(x > 0, x, a * x)
def sigmoid(x):            return 1.0 / (1.0 + np.exp(-x))

def _blocks(x, s=2):
    H, W = (x.shape[0] // s) * s, (x.shape[1] // s) * s
    return x[:H, :W].reshape(H // s, s, W // s, s)

def maxpool(x, s=2): return _blocks(x, s).max(axis=(1, 3))     # 2x2 window, stride 2
def avgpool(x, s=2): return _blocks(x, s).mean(axis=(1, 3))
def minpool(x, s=2): return _blocks(x, s).min(axis=(1, 3))

# ---- verification -------------------------------------------------------------
rng = np.random.default_rng(0)
x_test = rng.normal(size=(37, 41))
err_conv = max(np.abs(conv2d(x_test, k, pad='constant') -
                      correlate2d(x_test, k, mode='same', boundary='fill')).max()
               for k in KERNELS.values())
naive_max = np.array([[x_test[2*i:2*i+2, 2*j:2*j+2].max()  for j in range(20)] for i in range(18)])
naive_avg = np.array([[x_test[2*i:2*i+2, 2*j:2*j+2].mean() for j in range(20)] for i in range(18)])
print(f'conv2d  vs scipy.correlate2d : max |error| = {err_conv:.1e}')
print(f'maxpool vs naive loops       : max |error| = {np.abs(maxpool(x_test) - naive_max).max():.1e}')
print(f'avgpool vs naive loops       : max |error| = {np.abs(avgpool(x_test) - naive_avg).max():.1e}')
assert err_conv < 1e-10

In [ ]:
# Cross-check against PyTorch's own layers (torch is pre-installed on Colab)
try:
    import torch, torch.nn.functional as F
    xt = torch.tensor(x_test)[None, None]
    worst = 0.0
    for k in KERNELS.values():
        ours = conv2d(x_test, k, pad='constant')
        theirs = F.conv2d(xt, torch.tensor(k)[None, None], padding=1)[0, 0].numpy()
        worst = max(worst, np.abs(ours - theirs).max())
    mp = F.max_pool2d(xt, 2, 2)[0, 0].numpy(); ap = F.avg_pool2d(xt, 2, 2)[0, 0].numpy()
    print(f'conv2d  vs torch.nn.functional.conv2d    : {worst:.1e}')
    print(f'maxpool vs torch.nn.functional.max_pool2d: {np.abs(maxpool(x_test) - mp).max():.1e}')
    print(f'avgpool vs torch.nn.functional.avg_pool2d: {np.abs(avgpool(x_test) - ap).max():.1e}')
except ImportError:
    print('PyTorch not available here; SciPy check above is sufficient.')

## 1.4 Run the pipeline: Conv → (A) → ReLU → (B) → MaxPool → (C) / AvgPool → (D)

Each filter runs **independently** on the input image, and all four intermediate tensors are stored.

In [ ]:
def run_pipeline(image):
    out = {}
    for name, k in KERNELS.items():
        A = conv2d(image, k)      # Stage A: raw linear response
        B = relu(A)               # Stage B: ReLU
        C = maxpool(B)            # Stage C: MaxPool 2x2, stride 2
        D = avgpool(B)            # Stage D: AvgPool 2x2, stride 2
        out[name] = dict(A=A, B=B, C=C, D=D)
    return out

stages = run_pipeline(img)
probe_stages = run_pipeline(probe)
for name, S in stages.items():
    print(f"{SHORT[name]:8s} A{S['A'].shape} B{S['B'].shape} C{S['C'].shape} D{S['D'].shape}")

## 1.5 The 5 × 4 comparative visualization grid

**Intensity scaling rule (the same for every row):** each row uses one scale, *m* = max|Stage A| of that filter.
* **Stage A** uses a *diverging* map (blue = negative, white = 0, red = positive) on [−m, +m]. The sign is the whole story for ReLU, so it has to be visible.
* **Stages B, C, D** use grayscale on [0, m]. Black = exactly 0. Since all four columns share *m*, a pixel's brightness can be compared directly across stages. For example, an AvgPool value that looks dimmer than the MaxPool value really *is* smaller.

In [ ]:
STAGE_TITLES = {'A': 'Stage A · raw convolution', 'B': 'Stage B · ReLU',
                'C': 'Stage C · MaxPool 2×2 s2', 'D': 'Stage D · AvgPool 2×2 s2'}

def plot_grid(st, suptitle, fname, crop=None):
    fig, axes = plt.subplots(5, 4, figsize=(15, 19))
    for r, (name, S) in enumerate(st.items()):
        m = np.abs(S['A']).max()
        for c, key in enumerate('ABCD'):
            ax, M = axes[r, c], S[key]
            if crop is not None:                       # crop in full-res coords, halve for pooled maps
                y0, y1, x0, x1 = crop; f = 1 if key in 'AB' else 2
                M = M[y0 // f:y1 // f, x0 // f:x1 // f]
            if key == 'A':
                im = ax.imshow(M, cmap='RdBu_r', vmin=-m, vmax=m)
                extra = f'neg {100 * (S["A"] < 0).mean():.1f}%'
            else:
                im = ax.imshow(M, cmap='gray', vmin=0, vmax=m)
                extra = f'zeros {100 * (S[key] == 0).mean():.1f}%'
            ax.set_title(f'{STAGE_TITLES[key]}\n{S[key].shape[0]}×{S[key].shape[1]} | '
                         f'[{S[key].min():.2f}, {S[key].max():.2f}] | {extra}')
            ax.set_xticks([]); ax.set_yticks([])
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        axes[r, 0].set_ylabel(name, fontsize=11, fontweight='bold')
    fig.suptitle(suptitle, fontsize=14, y=0.995)
    plt.tight_layout(); savefig(fig, fname); plt.show()

plot_grid(stages, f'5×4 grid — {img_src}', '02_grid_main')

### Zoomed grid (64×64 crop)

At full size, single pixels are not visible. This crop of the same grid shows individual mortar lines, so a specific pixel can be pointed at during the viva. Pooled maps show the matching 32×32 region.

In [ ]:
CROP = (96, 160, 96, 160)    # y0, y1, x0, x1 in full-resolution coordinates
plot_grid(stages, f'Zoomed 5×4 grid — crop rows {CROP[0]}:{CROP[1]}, cols {CROP[2]}:{CROP[3]}',
          '03_grid_zoom', crop=CROP)

### Control experiment: same grid on the probe image

The answers can be predicted on paper first. H-edge on the rectangle: **top edge positive** (dark above, bright below → below − above > 0) and **bottom edge negative**, so the bottom edge should vanish after ReLU. Likewise the V-edge **left edge is positive and the right edge is negative**. The ramp has a constant gradient, so the V-edge gives a flat, non-zero band instead of a line.

In [ ]:
plot_grid(probe_stages, '5×4 grid on the probe image (predictions can be checked)', '04_grid_probe')

## 1.6 Quantitative evidence table

Numbers that the Part III answers refer to:

| column | meaning |
|---|---|
| `% neg (A)` | share of pixels with a negative raw response |
| `% zero (B)` | share set to exactly 0 by ReLU ("blackout") |
| `energy kept by ReLU %` | Σ B² / Σ A², the share of signal energy that survives ReLU |
| `peak kept Max/Avg %` | strongest pooled value ÷ strongest Stage-B value |
| `strong survive Max/Avg %` | of the 2×2 windows that contain a *strong* response (≥ 50 % of peak), the share that still read as strong after pooling |
| `HF energy Max/Avg` | high-frequency energy (variance of the Laplacian) of Stage C ÷ Stage D. > 1 means MaxPool kept more fine detail |

In [ ]:
LAPLACE = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], float)
def hf_energy(M): return np.var(conv2d(M, LAPLACE))

def strong_survival(B, pooled, frac=0.5):
    t = frac * B.max()
    has_strong = _blocks(B).max(axis=(1, 3)) >= t
    return 100 * (pooled[has_strong] >= t).mean()

def evidence_table(st):
    rows = []
    for name, S in st.items():
        A, B, C, D = S['A'], S['B'], S['C'], S['D']
        rows.append({
            'filter': SHORT[name],
            'A min': A.min(), 'A max': A.max(),
            '% neg (A)': 100 * (A < 0).mean(),
            '% zero (B)': 100 * (B == 0).mean(),
            'energy kept by ReLU %': 100 * (B ** 2).sum() / (A ** 2).sum(),
            'peak kept Max %': 100 * C.max() / B.max(),
            'peak kept Avg %': 100 * D.max() / B.max(),
            'strong survive Max %': strong_survival(B, C),
            'strong survive Avg %': strong_survival(B, D),
            'HF energy Max/Avg': hf_energy(C) / hf_energy(D),
        })
    return pd.DataFrame(rows).set_index('filter')

ev = evidence_table(stages)
ev_probe = evidence_table(probe_stages)
print('MAIN IMAGE'); display(ev.round(2))
print('PROBE IMAGE'); display(ev_probe.round(2))

# PART II — Architectural Comparison & Parameter Audit

## 2.1 Hand calculation (done *before* running `model.summary()`)

Formula: **Params(conv) = (K_h · K_w · C_in + 1) · C_out**. The "+1" is the one bias per output filter. Pooling and Global Average Pooling have **0** parameters (they only take a max or a mean). Dense: **(N_in + 1) · N_out**.

Input 32×32×3. GAP turns the last feature map (H×W×10) into a 10-vector, so the Dense layer always sees 10 inputs, **whatever the kernel size**.

### Model A (3×3 kernels, MaxPool)
| Layer | Calculation | Params |
|---|---|---|
| Conv1 3×3, 3→5 | (3·3·3 + 1)·5 = (27 + 1)·5 | **140** |
| MaxPool 2×2 | — | 0 |
| Conv2 3×3, 5→10 | (3·3·5 + 1)·10 = (45 + 1)·10 | **460** |
| MaxPool 2×2 | — | 0 |
| GAP | — | 0 |
| Dense 10→10 | (10 + 1)·10 | **110** |
| **Total** | 140 + 460 + 110 | **710** |

### Model B (5×5 kernels, AvgPool)
| Layer | Calculation | Params |
|---|---|---|
| Conv1 5×5, 3→5 | (5·5·3 + 1)·5 = (75 + 1)·5 | **380** |
| AvgPool 2×2 | — | 0 |
| Conv2 5×5, 5→10 | (5·5·5 + 1)·10 = (125 + 1)·10 | **1 260** |
| AvgPool 2×2 | — | 0 |
| GAP | — | 0 |
| Dense 10→10 | (10 + 1)·10 | **110** |
| **Total** | 380 + 1 260 + 110 | **1 750** |

**Shape trace ('valid' padding, Keras default)**
* A: 32 → conv3 → 30 → pool → 15 → conv3 → 13 → pool → 6 → GAP → 10 → Dense → 10
* B: 32 → conv5 → 28 → pool → 14 → conv5 → 10 → pool → 5 → GAP → 10 → Dense → 10

(Padding changes the *shapes* but never the *parameter count*. Weights depend only on K, C_in and C_out, not on H or W.)

The next cell re-derives the same numbers in code, step by step, so the hand calculation and the framework can be compared.

In [ ]:
def conv_params(k, cin, cout):  return (k * k * cin + 1) * cout
def dense_params(nin, nout):    return (nin + 1) * nout

SPECS = {'Model A (Standard)':     dict(k=3, pool='max'),
         'Model B (Large Kernel)': dict(k=5, pool='avg')}

manual = {}
for mname, s in SPECS.items():
    k = s['k']
    c1, c2, d = conv_params(k, 3, 5), conv_params(k, 5, 10), dense_params(10, 10)
    h1 = 32 - k + 1; p1 = h1 // 2; h2 = p1 - k + 1; p2 = h2 // 2
    print(f'--- {mname} ---')
    print(f'Conv1: ({k}x{k}x3 + 1) x 5  = ({k*k*3} + 1) x 5  = {c1:5d}   shape 32 -> {h1}')
    print(f'Pool1: 0 params                               shape {h1} -> {p1}')
    print(f'Conv2: ({k}x{k}x5 + 1) x 10 = ({k*k*5} + 1) x 10 = {c2:5d}   shape {p1} -> {h2}')
    print(f'Pool2: 0 params                               shape {h2} -> {p2}')
    print(f'GAP  : 0 params                               {p2}x{p2}x10 -> 10')
    print(f'Dense: (10 + 1) x 10             = {d:5d}')
    print(f'TOTAL = {c1} + {c2} + {d} = {c1 + c2 + d}\n')
    manual[mname] = dict(conv1=c1, conv2=c2, fc=d, total=c1 + c2 + d)

## 2.2 Framework verification — Keras `model.summary()`  *(screenshot this output for the report)*

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def build_keras(k, pool, name):
    P = layers.MaxPooling2D if pool == 'max' else layers.AveragePooling2D
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Conv2D(5, (k, k), activation='relu', name='conv1'),
        P(pool_size=2, strides=2, name='pool1'),
        layers.Conv2D(10, (k, k), activation='relu', name='conv2'),
        P(pool_size=2, strides=2, name='pool2'),
        layers.GlobalAveragePooling2D(name='gap'),
        layers.Dense(10, activation='softmax', name='fc'),
    ], name=name)

keras_models = {'Model A (Standard)': build_keras(3, 'max', 'Model_A_3x3_MaxPool'),
                'Model B (Large Kernel)': build_keras(5, 'avg', 'Model_B_5x5_AvgPool')}
code_counts = {}
for mname, m in keras_models.items():
    m.summary()
    code_counts[mname] = {l.name: l.count_params() for l in m.layers}
    code_counts[mname]['total'] = m.count_params()
    print()

In [ ]:
# Independent second framework: PyTorch
import torch, torch.nn as nn

def build_torch(k, pool):
    P = nn.MaxPool2d if pool == 'max' else nn.AvgPool2d
    return nn.Sequential(nn.Conv2d(3, 5, k), nn.ReLU(), P(2, 2),
                         nn.Conv2d(5, 10, k), nn.ReLU(), P(2, 2),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(10, 10))

torch_counts = {}
for mname, s in SPECS.items():
    tm = build_torch(s['k'], s['pool'])
    torch_counts[mname] = sum(p.numel() for p in tm.parameters() if p.requires_grad)
    out_shape = tuple(tm(torch.zeros(1, 3, 32, 32)).shape)
    print(f'{mname:24s} PyTorch trainable params = {torch_counts[mname]}   output {out_shape}')

## 2.3 Completed audit table

In [ ]:
audit_rows = []
for mname, s in SPECS.items():
    for lyr, conv_desc in [('conv1', 'Conv1: 5 filters'), ('conv2', 'Conv2: 10 filters'), ('fc', 'Dense 10 (after GAP)')]:
        man, code = manual[mname][lyr], code_counts[mname][lyr]
        audit_rows.append({'Model Configuration': mname, 'Layer': conv_desc,
                           'Kernel Size': f"{s['k']}×{s['k']}" if lyr != 'fc' else '—',
                           'Pooling Type': ('Max' if s['pool'] == 'max' else 'Avg') + ' Pooling (2×2, stride 2)',
                           'Manual Count': man, 'Code Output (Keras)': code,
                           'Match': '✅' if man == code else '❌'})
    audit_rows.append({'Model Configuration': mname, 'Layer': 'TOTAL', 'Kernel Size': '', 'Pooling Type': '',
                       'Manual Count': manual[mname]['total'],
                       'Code Output (Keras)': code_counts[mname]['total'],
                       'Match': '✅' if manual[mname]['total'] == code_counts[mname]['total'] == torch_counts[mname] else '❌'})
audit = pd.DataFrame(audit_rows)
display(audit)
print('Keras, PyTorch and hand calculation agree:',
      all(manual[m]['total'] == code_counts[m]['total'] == torch_counts[m] for m in SPECS))

## 2.4 Receptive field: formula **and** measurement

Formula (applied layer by layer): $r_{out} = r_{in} + (k-1)\,j_{in}$, $\;j_{out} = j_{in}\cdot s$ (r = receptive field, j = jump/stride product).

To check this a different way, we **measure** it. Build each network with all weights = 1, no activation, avg-pooling. Backpropagate from the centre output pixel and count how many input pixels receive a non-zero gradient. That support size is the receptive field.

Model C (stacked 3×3) replaces every 5×5 conv of Model B with two 3×3 convs. Its receptive field should match Model B exactly.

In [ ]:
ARCHS = {   # sequence of ('conv', k) / ('pool', 2)
    'Model A (3×3)':           [('conv', 3), ('pool', 2), ('conv', 3), ('pool', 2)],
    'Model B (5×5)':           [('conv', 5), ('pool', 2), ('conv', 5), ('pool', 2)],
    'Model C (two 3×3 per 5×5)': [('conv', 3), ('conv', 3), ('pool', 2), ('conv', 3), ('conv', 3), ('pool', 2)],
    'single 5×5 layer':        [('conv', 5)],
    'two stacked 3×3 layers':  [('conv', 3), ('conv', 3)],
}

def rf_formula(arch):
    r, j = 1, 1
    for kind, k in arch:
        r, j = r + (k - 1) * j, j * (k if kind == 'pool' else 1)
    return r

for a, s in ARCHS.items():
    print(f'{a:28s} RF (formula) = {rf_formula(s)}')

In [ ]:
def rf_measured(arch, n=64):
    x = torch.zeros(1, 1, n, n, requires_grad=True)
    h = x
    for kind, k in arch:
        if kind == 'conv':
            conv = nn.Conv2d(1, 1, k, bias=False); nn.init.ones_(conv.weight); h = conv(h)
        else:
            h = nn.functional.avg_pool2d(h, k)
    c = h.shape[-1] // 2
    h[0, 0, c, c].backward()
    rows = (x.grad[0, 0].abs().sum(1) > 0).nonzero()
    return int(rows.max() - rows.min() + 1)

rf_tab = pd.DataFrame([{'architecture': a, 'RF formula': rf_formula(s), 'RF measured (gradient)': rf_measured(s)}
                       for a, s in ARCHS.items()]).set_index('architecture')
display(rf_tab)

## 2.5 Compute cost (multiply-accumulates)

Parameters are only half the cost. A conv layer does **H_out · W_out · K² · C_in · C_out** MACs, so a larger kernel costs more *per output pixel* as well as more in weights.

In [ ]:
def macs(k):
    h1 = 32 - k + 1; p1 = h1 // 2; h2 = p1 - k + 1
    return h1 * h1 * k * k * 3 * 5 + h2 * h2 * k * k * 5 * 10 + 10 * 10
cost = pd.DataFrame({'params': [manual[m]['total'] for m in SPECS],
                     'MACs / image': [macs(3), macs(5)],
                     'RF (pixels)': [rf_formula(ARCHS['Model A (3×3)']), rf_formula(ARCHS['Model B (5×5)'])]},
                    index=list(SPECS))
display(cost)

# PART III — Critical Thinking: experiments, then answers

Each question has (a) the experiment code and figure, then (b) the written answer, which cites the numbers printed above it.

> The numbers quoted in the answers are for the default brick-wall image. If you switch to your own image (`USE_MY_OWN_IMAGE = True`), re-read them from the evidence tables and update the answers. The *reasoning* stays the same.

---
## Q1 · The ReLU "blackout" effect & activation alternatives

### Experiment 1a — where exactly do pixels go black?
Red overlay = pixels where Stage A < 0, i.e. the pixels ReLU sets to 0.

In [ ]:
H_NAME, V_NAME = list(KERNELS)[0], list(KERNELS)[1]

fig, axes = plt.subplots(2, 4, figsize=(17, 8.5))
for r, (nm, image, st) in enumerate([('main', img, stages), ('probe', probe, probe_stages)]):
    for c, key in enumerate([H_NAME, V_NAME]):
        A = st[key]['A']
        m = np.abs(A).max()
        axes[r, 2 * c].imshow(A, cmap='RdBu_r', vmin=-m, vmax=m)
        axes[r, 2 * c].set_title(f'{SHORT[key]} Stage A ({nm})')
        ov = np.dstack([image] * 3)
        ov[A < -0.05 * m] = [1, 0, 0]                    # clearly negative -> red
        axes[r, 2 * c + 1].imshow(ov)
        axes[r, 2 * c + 1].set_title(f'{SHORT[key]}: red = blacked out by ReLU\n'
                                     f'{100 * (A < 0).mean():.1f}% of pixels negative')
for a in axes.ravel(): a.axis('off')
plt.tight_layout(); savefig(fig, '05_q1_blackout_overlay'); plt.show()

# Polarity check on the probe rectangle (rows 24:64, cols 24:104)
Ah, Av = probe_stages[H_NAME]['A'], probe_stages[V_NAME]['A']
print('Probe rectangle, H-edge response at TOP edge    (row 23/24, col 64):', Ah[23, 64], Ah[24, 64])
print('Probe rectangle, H-edge response at BOTTOM edge (row 63/64, col 64):', Ah[63, 64], Ah[64, 64])
print('Probe rectangle, V-edge response at LEFT edge   (row 44, col 23/24):', Av[44, 23], Av[44, 24])
print('Probe rectangle, V-edge response at RIGHT edge  (row 44, col 103/104):', Av[44, 103], Av[44, 104])

### Experiment 1b — swap the activation: ReLU vs Leaky ReLU (α = 0.1) vs Sigmoid
Plotted on the same scale as Stage A. Below the maps, a **1-D intensity profile** down one image column shows each function's effect on the same edge responses.

In [ ]:
ACTS = {'ReLU': relu, 'Leaky ReLU (α=0.1)': leaky_relu, 'Sigmoid': sigmoid}

fig, axes = plt.subplots(2, 4, figsize=(17, 8.5))
act_rows = []
for r, key in enumerate([H_NAME, V_NAME]):
    A = stages[key]['A']; m = np.abs(A).max()
    axes[r, 0].imshow(A, cmap='RdBu_r', vmin=-m, vmax=m); axes[r, 0].set_title(f'{SHORT[key]} · Stage A (raw)')
    for c, (an, f) in enumerate(ACTS.items(), start=1):
        Y = f(A)
        if an == 'Sigmoid':
            im = axes[r, c].imshow(Y, cmap='gray', vmin=0, vmax=1)
        else:
            im = axes[r, c].imshow(Y, cmap='RdBu_r', vmin=-m, vmax=m)
        axes[r, c].set_title(f'{SHORT[key]} · {an}\nrange [{Y.min():.2f}, {Y.max():.2f}]')
        fig.colorbar(im, ax=axes[r, c], fraction=0.046)
        act_rows.append({'filter': SHORT[key], 'activation': an,
                         '% exactly zero': 100 * (Y == 0).mean(),
                         'value on flat region (A=0)': float(f(np.array(0.0))),
                         'output range': f'[{Y.min():.3f}, {Y.max():.3f}]',
                         'sign of input recoverable?': 'no' if an == 'ReLU' else 'yes',
                         'mean |gradient| dY/dA': float(np.mean(np.where(A > 0, 1, 0)) if an == 'ReLU' else
                                                        np.mean(np.where(A > 0, 1, 0.1)) if an.startswith('Leaky') else
                                                        np.mean(sigmoid(A) * (1 - sigmoid(A))))})
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); savefig(fig, '06_q1_activation_swap'); plt.show()
display(pd.DataFrame(act_rows).set_index(['filter', 'activation']).round(3))

# 1-D profile through one column of the H-edge map
col, y0, y1 = IMG_SIZE // 2, 0, 110
A = stages[H_NAME]['A'][y0:y1, col]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(img[y0:y1, col] * np.abs(A).max(), color='0.75', lw=3, label='input intensity (scaled)')
ax.plot(A, 'k-', lw=1.2, label='Stage A raw')
ax.plot(relu(A), 'r-', lw=1.6, label='ReLU')
ax.plot(leaky_relu(A), 'b--', lw=1.2, label='Leaky ReLU α=0.1')
ax.plot(sigmoid(A), 'g-', lw=1.2, label='Sigmoid')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel(f'row (column {col} of the image)'); ax.set_ylabel('value')
ax.set_title('H-edge response down one column: each bright mortar line gives a + lobe (top) and a − lobe (bottom)')
ax.legend(ncol=5, fontsize=8); plt.tight_layout(); savefig(fig, '07_q1_profile'); plt.show()

### ✍️ Answer — Q1

**Which regions turned black?** *(H-edge and V-edge rows, Stage A → Stage B)*

* **Horizontal-edge row:** each horizontal mortar line is a thin **bright** stripe. The kernel computes *(3 pixels below) − (3 pixels above)*. On the **upper boundary** of a mortar line the pixels below are bright → **positive** (red in Stage A, kept). On the **lower boundary** the pixels below are dark brick → **negative** (blue in Stage A) → **black in Stage B**. So in Stage B **every mortar line keeps only its top edge, and the bottom edge disappears**. The 1-D profile (Fig. 07) shows this as a +/− lobe pair at each line, with ReLU cutting off the − lobe. The flat interior of each brick is ≈ 0 before ReLU and exactly 0 after it.
* **Vertical-edge row:** the same thing happens horizontally. Sobel-x computes *(right column) − (left column)*. The **left** side of each vertical joint is positive and kept; the **right** side is negative and blacked out.
* **Size of the effect (table 1.6):** on the main image about **49.9% (H-edge)** and **43.9% (V-edge)** of pixels are negative in Stage A, and ReLU keeps only **49.7% / 47.1%** of the response energy. For comparison, Sharpen and Blur lose only 0.02% / 0.0%. Emboss loses 11.2%: its sum is 1, but its strong −2 corner makes the 'shadow' side of each edge (bright on the upper-left, dark on the lower-right) negative, which is why emboss shows thin black lines on one side of every mortar joint in Stage B.
* **Proof on the probe image:** the rectangle's top edge gives **+3** and its bottom edge **−3** (H-edge), and the left edge gives **+4**, the right edge **−4** (V-edge) (printout of Exp. 1a). After ReLU, the bottom and right edges of the rectangle are **gone**, exactly as predicted.

**Why were they negative?** Both edge kernels have **sum = 0**: they are *derivative* operators. A derivative is signed. It is + where brightness rises in the kernel's direction (dark→bright) and − where it falls (bright→dark). Every bright stripe has one rising and one falling side, so about half of all edge evidence is negative by construction. Sharpen, blur and emboss have **sum = 1**, so their outputs sit around the local brightness (≥ 0), and almost nothing is removed.

**What if the activation changes?** (Fig. 06 and table)
* **Leaky ReLU (α = 0.1):** f(x) = x for x > 0, 0.1x otherwise. The bottom/right edges **reappear as faint negative lines** at 1/10 strength instead of being deleted. Mathematically the map is *invertible* (x = f(x)/0.1 for negatives), so the polarity information is kept. The gradient there is 0.1 instead of 0, so these units cannot "die". The share of exact zeros drops from **50.7%** to **0.8%** (only pixels where A is exactly 0).
* **Sigmoid:** σ(x) = 1/(1+e^{−x}). Nothing becomes black. **Flat regions (A = 0) map to 0.5 (mid-gray)**, positive edges to > 0.5 (lighter), negative edges to < 0.5 (darker). So the map looks like a low-contrast, gray version of Stage A: both polarities are kept but squashed into (0, 1). Its derivative is at most 0.25 (mean |dσ/dA| ≈ 0.24 here), so gradients shrink by ≥ 4× per layer (vanishing gradients). Its output is also never zero-centred, which slows training. This is why ReLU-family activations replaced sigmoid inside CNNs.

**Design takeaway:** ReLU after a zero-sum edge filter *halves* the edge information. A learned CNN compensates by learning **both polarities as separate filters** (e.g. one +∂/∂y and one −∂/∂y), so layer-1 filters often come in opposite-sign pairs.

---
## Q2 · Max pooling vs average pooling: information retention

### Experiment 2a — side by side on Sharpen and edge maps (zoomed)

In [ ]:
names_q2 = [list(KERNELS)[2], H_NAME, V_NAME]
y0, y1, x0, x1 = CROP
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for r, key in enumerate(names_q2):
    S = stages[key]; m = np.abs(S['A']).max()
    B = S['B'][y0:y1, x0:x1]; C = S['C'][y0//2:y1//2, x0//2:x1//2]; D = S['D'][y0//2:y1//2, x0//2:x1//2]
    for c, (M, t) in enumerate([(B, 'Stage B (before pooling)'), (C, 'Stage C MaxPool'), (D, 'Stage D AvgPool')]):
        axes[r, c].imshow(M, cmap='gray', vmin=0, vmax=m); axes[r, c].set_title(f'{SHORT[key]} · {t}')
    im = axes[r, 3].imshow(C - D, cmap='magma', vmin=0); axes[r, 3].set_title('C − D  (detail lost by AvgPool)')
    fig.colorbar(im, ax=axes[r, 3], fraction=0.046)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); savefig(fig, '08_q2_max_vs_avg'); plt.show()
display(ev[['peak kept Max %', 'peak kept Avg %', 'strong survive Max %', 'strong survive Avg %', 'HF energy Max/Avg']].round(2))

### Experiment 2b — dark ink text on a white background (pixels ≈ 255)
Pooling is applied directly to the pixels, 1×, 2× and 3× in a row (downsampling ×2, ×4, ×8). Two extra options are added: **MinPool** and the standard trick **invert → MaxPool → invert** (what MNIST does: white strokes on a black background).

*Stroke retention* = (share of ink pixels < 0.5 after pooling) ÷ (share before). 1.0 = text kept at the right thickness, 0 = text erased, > 1 = text thickened.

In [ ]:
# A tiny built-in 5x7 bitmap font -> identical result on every machine / OpenCV version.
# Each font pixel becomes a 3x3 block, so ink strokes are 3 px wide (like thin pen strokes in a scan),
# and the text starts at an odd offset so strokes do not line up neatly with the 2x2 pooling grid.
GLYPHS = {
 'C': ['.###.', '#...#', '#....', '#....', '#....', '#...#', '.###.'],
 'N': ['#...#', '##..#', '#.#.#', '#..##', '#...#', '#...#', '#...#'],
 'P': ['####.', '#...#', '#...#', '####.', '#....', '#....', '#....'],
 'O': ['.###.', '#...#', '#...#', '#...#', '#...#', '#...#', '.###.'],
 'L': ['#....', '#....', '#....', '#....', '#....', '#....', '#####'],
 'I': ['.###.', '..#..', '..#..', '..#..', '..#..', '..#..', '.###.'],
 'G': ['.###.', '#...#', '#....', '#.###', '#...#', '#...#', '.###.'],
 ' ': ['.....'] * 7}

def render_text(text, stroke=3, offset=(5, 7)):
    glyph_cols = []
    for ch in text:
        glyph_cols.append(np.array([[c == '#' for c in row] for row in GLYPHS[ch]]))
        glyph_cols.append(np.zeros((7, 1), bool))                      # 1-pixel letter spacing
    bitmap = np.hstack(glyph_cols)
    ink = np.kron(bitmap, np.ones((stroke, stroke), bool))             # upscale: 1 px -> stroke x stroke
    page = np.ones((ink.shape[0] + 2 * offset[0] + 1, ink.shape[1] + 2 * offset[1] + 1))   # white = 1.0 (=255)
    page[offset[0]:offset[0] + ink.shape[0], offset[1]:offset[1] + ink.shape[1]][ink] = 0.0   # black ink
    return page

T0 = render_text('CNN POOLING')
ink0 = (T0 < 0.5).mean()

POOLS = {'MaxPool': maxpool, 'AvgPool': avgpool, 'MinPool': minpool,
         'invert→MaxPool→invert': lambda x: 1 - maxpool(1 - x)}
fig, axes = plt.subplots(len(POOLS) + 1, 3, figsize=(15, 11))
for c in range(3):
    axes[0, c].imshow(T0, cmap='gray', vmin=0, vmax=1); axes[0, c].set_title('original (white = 1.0 = 255)')
ret_rows = []
for r, (pn, f) in enumerate(POOLS.items(), start=1):
    X = T0
    for lvl in range(1, 4):
        X = f(X)
        ret = (X < 0.5).mean() / ink0
        axes[r, lvl - 1].imshow(X, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        axes[r, lvl - 1].set_title(f'{pn} ×{lvl} ({X.shape[0]}×{X.shape[1]}) | stroke retention {ret:.2f}')
        ret_rows.append({'pooling': pn, 'levels': lvl, 'size': f'{X.shape[0]}×{X.shape[1]}',
                         'stroke retention': ret, 'darkest pixel': X.min()})
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); savefig(fig, '09_q2_text_on_white'); plt.show()
ret_df = pd.DataFrame(ret_rows).pivot(index='pooling', columns='levels', values='stroke retention')
ret_df.columns = [f'after {c}× pooling' for c in ret_df.columns]
display(ret_df.round(3))

### ✍️ Answer — Q2

**What MaxPool kept that AvgPool washed out** (Sharpen, H-edge, V-edge rows, Stage C vs D, Fig. 08)

* The mortar lines are only **1–2 pixels wide** in the edge and sharpen maps. A 2×2 window usually contains one or two bright "line" pixels and two or three near-zero "brick" pixels.
  * **MaxPool** outputs the bright value → the line stays at **full intensity** and stays continuous. Peak kept: **100%** for every filter.
  * **AvgPool** divides it by the number of pixels: a line that fills one of four pixels drops to ¼ of its value → lines become **thinner, dimmer and broken**. Peak kept with AvgPool: **83.9% (Sharpen), 85.0% (H-edge), 81.7% (V-edge)**.
* *Strong-response survival*: of the windows that contained a strong edge pixel, MaxPool keeps **100%** of them strong. AvgPool keeps only **63.6% / 35.5% / 40.1%** (Sharpen / H / V).
* *High-frequency energy* of the MaxPool map is **1.24× / 1.82× / 1.77×** that of the AvgPool map. The C − D panel shows the lost detail sits exactly on the thin edges and the sharpen halo, while flat brick areas give C ≈ D.
* Features preserved by Max but washed out by Avg: **thin mortar lines, mortar-joint corners/intersections, the bright halos the Sharpen kernel puts around each edge, and isolated small texture specks on the brick surface.**
* For the low-pass **Blur** row the two are nearly identical (HF ratio ≈ 1.13), because there is no fine detail left to lose.

**Dark text (≈ 0) on white (≈ 255): which pooling preserves the strokes?** (Fig. 09)

* **Average pooling preserves the text better; MaxPool erases it.** MaxPool returns the *brightest* value in each window. Any window that touches even one white background pixel outputs white (255). Ink strokes are only about 3 px wide, so most windows contain some background, and MaxPool deletes the strokes. Stroke retention falls to **0.62 → 0.00 → 0.00** after 1, 2, 3 poolings: the letters thin out, break apart and finally disappear.
* AvgPool mixes ink and background, so strokes become gray and blurred but **remain visible** (retention **0.64 → 0.67 → 0.29**; the darkest pixel is still 0.00 → 0.13 → 0.38, i.e. clearly dark). At ×8 downsampling the image is only 4×26 pixels, so any method blurs it, but AvgPool still shows *where* the ink is while MaxPool shows a blank page.
* **Better still** (not one of the two options, but the right engineering answer): flip the polarity. Either use **MinPool** (= −MaxPool(−x)), or **invert the image so ink is bright on a dark background and then MaxPool**. Both give retention **1.41 → 2.24 → 4.37**: every stroke survives, but strokes get thicker each time and letters start to merge ("CNN" turns into a black block at ×8). So min/inverted-max is best for 1–2 pooling steps; for deeper downsampling, a learned strided convolution is safer. This is why MNIST is stored as white digits on black. **Rule: MaxPool keeps whatever is *bright*, so the feature you care about must be the high-valued one.** Inside a trained CNN this usually happens automatically: a conv filter can learn a negative weight on dark ink, so after ReLU the strokes become *bright* activations, and then MaxPool is the right choice again.

---
## Q3 · Parameter explosion & receptive-field equivalence

In [ ]:
pa, pb = manual['Model A (Standard)']['total'], manual['Model B (Large Kernel)']['total']
ca = manual['Model A (Standard)']['conv1'] + manual['Model A (Standard)']['conv2']
cb = manual['Model B (Large Kernel)']['conv1'] + manual['Model B (Large Kernel)']['conv2']
print(f'Total params : A = {pa}, B = {pb}, increase = {pb - pa} -> ({pb} - {pa}) / {pa} x 100 = {100 * (pb - pa) / pa:.2f} %')
print(f'Conv only    : A = {ca}, B = {cb}, increase = {cb - ca} -> {100 * (cb - ca) / ca:.2f} %')
print(f'Kernel-area ratio 25/9 = {25 / 9:.3f}  (weights grow x{25/9:.2f}; biases and the Dense layer do not grow)')
print(f'MACs per image: A = {macs(3):,}, B = {macs(5):,}  (x{macs(5) / macs(3):.2f})')

### Experiment 3 — replace each 5×5 by two 3×3: does it always save parameters?

Two stacked 3×3 convolutions have RF = 1 + 2 + 2 = **5** (measured above: 5 ✔). The parameter count depends on the **width of the middle layer** (C_mid):

$$P_{5\times5} = (25\,C_{in}+1)\,C_{out} \qquad P_{3\times3+3\times3} = (9\,C_{in}+1)\,C_{mid} + (9\,C_{mid}+1)\,C_{out}$$

In [ ]:
def stacked(cin, cmid, cout): return conv_params(3, cin, cmid) + conv_params(3, cmid, cout)
rows = []
for where, cin, cout in [('Model-B conv1 (3→5)', 3, 5), ('Model-B conv2 (5→10)', 5, 10), ('textbook case C→C, C=64', 64, 64)]:
    p5 = conv_params(5, cin, cout)
    for label, cmid in [('C_mid = C_in', cin), ('C_mid = C_out', cout)]:
        p33 = stacked(cin, cmid, cout)
        rows.append({'layer': where, 'middle width': f'{label} ({cmid})', 'one 5×5': p5, 'two 3×3': p33,
                     'saved': p5 - p33, 'saved %': 100 * (p5 - p33) / p5})
display(pd.DataFrame(rows).set_index(['layer', 'middle width']).round(1))

In [ ]:
# Whole-model check in Keras
def build_keras_stacked(cmid_mode, name):
    m1 = 3 if cmid_mode == 'in' else 5
    m2 = 5 if cmid_mode == 'in' else 10
    return keras.Sequential([keras.Input((32, 32, 3)),
        layers.Conv2D(m1, 3, activation='relu'), layers.Conv2D(5, 3, activation='relu'), layers.AveragePooling2D(2),
        layers.Conv2D(m2, 3, activation='relu'), layers.Conv2D(10, 3, activation='relu'), layers.AveragePooling2D(2),
        layers.GlobalAveragePooling2D(), layers.Dense(10, activation='softmax')], name=name)

mc_in = build_keras_stacked('in', 'Model_C_stacked3x3_mid_eq_Cin')
mc_out = build_keras_stacked('out', 'Model_C_stacked3x3_mid_eq_Cout')
print(f'Model B (5x5)                 : {pb} params, RF {rf_formula(ARCHS["Model B (5×5)"])}')
print(f'Model C, middle width = C_in  : {mc_in.count_params()} params, RF {rf_formula(ARCHS["Model C (two 3×3 per 5×5)"])}'
      f'  -> saves {pb - mc_in.count_params()} ({100 * (pb - mc_in.count_params()) / pb:.1f}%)')
print(f'Model C, middle width = C_out : {mc_out.count_params()} params, RF {rf_formula(ARCHS["Model C (two 3×3 per 5×5)"])}'
      f'  -> saves {pb - mc_out.count_params()} ({100 * (pb - mc_out.count_params()) / pb:.1f}%)')

### ✍️ Answer — Q3

**Percentage increase A → B**

$$\frac{1750-710}{710}\times100 = \frac{1040}{710}\times100 = \mathbf{146.48\%}$$

Model B has **2.46×** the parameters of Model A. Counting only the conv layers (600 → 1 640) the increase is **173.33%**, close to the kernel-area ratio 25/9 = 2.78×. It is not exactly 2.78× because the biases (5 + 10) do not grow with kernel size, and the Dense layer (110) is identical in both thanks to GAP. Compute grows too: **197 650 → 419 100 MACs per image (×2.12)**.

**How many 3×3 layers equal one 5×5 receptive field?** **Two.** Each 3×3 adds k − 1 = 2 pixels: 1 + 2 + 2 = 5. This is confirmed by formula and by gradient measurement (table 2.4: "two stacked 3×3" → RF = 5). The full Model C (two 3×3 per 5×5) has RF = **16**, identical to Model B.

**How many parameters are saved?**
* **Standard (textbook) case, C_in = C_mid = C_out = C:** one 5×5 = 25C² + C, two 3×3 = 18C² + 2C → saving **7C² − C ≈ 28%**. For C = 64 that is 102 464 vs 73 856 → **28 608 fewer parameters (27.9%)**.
* **In this assignment's Model B (my extra finding):** the saving depends on the middle width.
  * Middle width = C_in → conv1: 380 → 224, conv2: 1 260 → 690. Whole model **1 750 → 1 024: saves 726 parameters (41.5%)**.
  * Middle width = C_out (the "VGG habit") → conv1 saves only 10 (380 → 370), and conv2 *costs more* (1 260 → 1 370). Whole model **1 750 → 1 850: 100 parameters MORE**.
  * Reason: with C_mid = C_out the second 3×3 layer is C_out × C_out. When the layer *doubles* the channels (5 → 10), that 10×10 layer is more expensive than the 5×5 it replaced. **"Two 3×3 is cheaper" holds only if the middle layer is not wider than the input.**
* Extra benefit regardless of count: two layers mean **two ReLUs**, so the stacked version is *more non-linear* (more expressive) for the same receptive field. This was the main argument of the VGG paper.

---
## Q4 · Operation commutativity: Conv → ReLU → MaxPool vs Conv → MaxPool → ReLU

In [ ]:
def show(P):
    return '\n'.join('    ' + '  '.join(f'{v:5.1f}' for v in row) for row in P)

patches = {'all negative': np.array([[-3., -1.], [-2., -5.]]),
           'mixed signs': np.array([[-3., 2.], [1., -4.]]),
           'avg-pool counterexample': np.array([[-3., 1.], [2., -4.]])}
for nm, P in patches.items():
    print(f'Patch ({nm}):\n{show(P)}')
    print(f'  MaxPool(ReLU(P)) = max({relu(P).ravel().tolist()}) = {relu(P).max():.1f}')
    print(f'  ReLU(MaxPool(P)) = ReLU({P.max():.1f})            = {relu(P.max()):.1f}')
    print(f'  AvgPool(ReLU(P)) = {relu(P).mean():.2f}   vs   ReLU(AvgPool(P)) = {relu(P.mean()):.2f}\n')

# Check on all 5 real feature maps
print('Whole-map check on the five Stage-A maps of the main image:')
for name, S in stages.items():
    A = S['A']
    same_max = np.array_equal(maxpool(relu(A)), relu(maxpool(A)))
    d_avg = np.abs(avgpool(relu(A)) - relu(avgpool(A))).max()
    same_leaky = np.allclose(maxpool(leaky_relu(A)), leaky_relu(maxpool(A)))
    same_sig = np.allclose(maxpool(sigmoid(A)), sigmoid(maxpool(A)))
    print(f'  {SHORT[name]:8s} Max: identical={same_max} | Leaky: {same_leaky} | Sigmoid: {same_sig} | '
          f'Avg: max diff = {d_avg:.3f}')

# Efficiency benchmark on a realistic activation tensor (64 channels x 256 x 256)
X = np.random.default_rng(1).normal(size=(64, 256, 256)).astype(np.float32)
def mp_c(x): return x.reshape(64, 128, 2, 128, 2).max(axis=(2, 4))
t_relu_first = min(timeit.repeat(lambda: mp_c(np.maximum(X, 0)), number=5, repeat=3)) / 5
t_pool_first = min(timeit.repeat(lambda: np.maximum(mp_c(X), 0), number=5, repeat=3)) / 5
print(f'\nReLU on {X.size:,} values  then pool : {1e3 * t_relu_first:.1f} ms')
print(f'Pool first, ReLU on {X.size // 4:,} values : {1e3 * t_pool_first:.1f} ms   '
      f'(ReLU work cut by 75%; measured speed-up x{t_relu_first / t_pool_first:.2f})')
print('Identical outputs:', np.array_equal(mp_c(np.maximum(X, 0)), np.maximum(mp_c(X), 0)))

### ✍️ Answer — Q4

**The outputs are *identical*.** ReLU and MaxPool commute.

**Proof.** ReLU(x) = max(0, x) is **monotonically non-decreasing**: a ≤ b ⇒ ReLU(a) ≤ ReLU(b). So the element that is largest before ReLU is still the largest after it, and for any window W:

$$\max_{i\in W}\,\max(0,x_i) \;=\; \max\Big(0,\ \max_{i\in W} x_i\Big)$$

because both sides equal the largest of {0, x₁, x₂, x₃, x₄} (max is associative and commutative).

**2×2 example with negative numbers** (printed above):

| patch | Conv → ReLU → MaxPool | Conv → MaxPool → ReLU |
|---|---|---|
| [[−3, −1], [−2, −5]] | ReLU → [[0,0],[0,0]] → max = **0** | max = −1 → ReLU(−1) = **0** |
| [[−3, 2], [1, −4]] | ReLU → [[0,2],[1,0]] → max = **2** | max = 2 → ReLU(2) = **2** |

On all five full feature maps the two orders give **bit-identical** tensors (`np.array_equal = True`). The same holds for Leaky ReLU and Sigmoid, which are also monotonic.

**Computational implication.** Pooling first means ReLU runs on **¼ of the values** (2×2, stride 2). On a 64×256×256 tensor that is 4.19 M → 1.05 M operations, 75% fewer, and a measured wall-clock speed-up shown in the printout (it depends on the machine; the ReLU itself is exactly 4× cheaper, while the pooling cost stays the same, so the pair speeds up by less than 4×). The backward pass is unchanged as well: the gradient goes to the arg-max element in both orders and is masked by ReLU in both orders. This reordering is a free optimisation that inference compilers (e.g. TensorRT, XLA) can apply.

**Warning, it does not hold for every pooling:** with **AvgPool** the order matters. For the patch [[−3, 1], [2, −4]]: AvgPool(ReLU) = (0+1+2+0)/4 = **0.75**, but ReLU(AvgPool) = ReLU(−1) = **0**. Averaging is linear and ReLU is not, so the negatives cancel the positives before ReLU can remove them. On the real maps the difference reaches 0.49. So the developer's change is safe for MaxPool **only**.

---
## Q5 · Diagnosing filter redundancy (filter collapse)

### Experiment 5 — reproduce the bug, then fix it
Model A is trained (PyTorch) for **20 epochs on a CIFAR-10 subset**, the same setup as in the scenario. Every epoch we measure:
* **filter similarity**: mean pairwise cosine similarity of the five Layer-1 weight vectors,
* **feature-map similarity**: mean pairwise *Pearson correlation* of the five Layer-1 activation maps on 256 test images (two all-zero maps are counted as identical = 1),
* **dead filters**: Layer-1 filters whose map is constant (for ReLU: 0 everywhere) on all 256 test images. Such a map carries no information and passes back no gradient.

| Run | What is broken / fixed |
|---|---|
| ① Constant init (bug) | all conv weights = 0.05, biases = 0 |
| ② He init (fix for ①) | `kaiming_normal_` random weights |
| ③ Dead ReLU (bug) | conv1 bias = −20, so every pre-activation < 0 |
| ④ BatchNorm (fix for ③) | same bad bias, but Conv → **BN** → ReLU |
| ⑤ Leaky ReLU (fix for ③) | same bad bias, ReLU → LeakyReLU(0.1) |

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
N_TRAIN, N_TEST, EPOCHS, BS = 6000, 2000, 20, 128
print('device:', dev)

def get_data():
    try:
        tfm = T.Compose([T.ToTensor(), T.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))])
        tr = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=tfm)
        te = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tfm)
        g = torch.Generator().manual_seed(0)
        itr = torch.randperm(len(tr), generator=g)[:N_TRAIN]; ite = torch.randperm(len(te), generator=g)[:N_TEST]
        Xtr = torch.stack([tr[i][0] for i in itr]); ytr = torch.tensor([tr[i][1] for i in itr])
        Xte = torch.stack([te[i][0] for i in ite]); yte = torch.tensor([te[i][1] for i in ite])
        return Xtr, ytr, Xte, yte, 'CIFAR-10 subset'
    except Exception as e:
        print('CIFAR-10 download failed, using a synthetic fallback:', e)
        g = torch.Generator().manual_seed(0)
        X = torch.randn(N_TRAIN + N_TEST, 3, 32, 32, generator=g)
        y = torch.randint(0, 10, (N_TRAIN + N_TEST,), generator=g)
        return X[:N_TRAIN], y[:N_TRAIN], X[N_TRAIN:], y[N_TRAIN:], 'synthetic'

Xtr, ytr, Xte, yte, dname = get_data()
Xtr, ytr, Xte, yte = Xtr.to(dev), ytr.to(dev), Xte.to(dev), yte.to(dev)
print(dname, Xtr.shape, Xte.shape)

class ModelA(nn.Module):
    def __init__(self, act='relu', bn=False):
        super().__init__()
        self.conv1, self.conv2 = nn.Conv2d(3, 5, 3), nn.Conv2d(5, 10, 3)
        self.bn1 = nn.BatchNorm2d(5) if bn else nn.Identity()
        self.bn2 = nn.BatchNorm2d(10) if bn else nn.Identity()
        self.act = nn.ReLU() if act == 'relu' else nn.LeakyReLU(0.1)
        self.fc = nn.Linear(10, 10)
    def forward(self, x, return_a1=False):
        a1 = self.act(self.bn1(self.conv1(x)))
        h = F.max_pool2d(a1, 2)
        h = F.max_pool2d(self.act(self.bn2(self.conv2(h))), 2)
        out = self.fc(h.mean(dim=(2, 3)))
        return (out, a1) if return_a1 else out

def make_run(run):
    torch.manual_seed(0)
    if run == '① Constant init (bug)':
        m = ModelA()
        for c in (m.conv1, m.conv2): nn.init.constant_(c.weight, 0.05); nn.init.zeros_(c.bias)
    elif run == '② He init (fix)':
        m = ModelA()
        for c in (m.conv1, m.conv2): nn.init.kaiming_normal_(c.weight, nonlinearity='relu'); nn.init.zeros_(c.bias)
    elif run == '③ Dead ReLU (bug)':
        m = ModelA(); nn.init.constant_(m.conv1.bias, -20.0)
    elif run == '④ BatchNorm (fix)':
        m = ModelA(bn=True); nn.init.constant_(m.conv1.bias, -20.0)
    elif run == '⑤ Leaky ReLU (fix)':
        m = ModelA(act='leaky'); nn.init.constant_(m.conv1.bias, -20.0)
    return m.to(dev)

@torch.no_grad()
def diagnostics(m):
    m.eval()
    _, a1 = m(Xte[:256], return_a1=True)
    W = F.normalize(m.conv1.weight.flatten(1), dim=1)
    wsim = W @ W.T
    A = a1.transpose(0, 1).flatten(1)                      # (5 filters, all pixels)
    Ac = A - A.mean(dim=1, keepdim=True)
    std = Ac.norm(dim=1)
    const = std < 1e-8          # constant map (e.g. all zeros) = dead filter: carries no information, gets no useful gradient
    dead = const
    fsim = (Ac @ Ac.T) / (std[:, None] * std[None, :] + 1e-12)
    fsim[const[:, None] & const[None, :]] = 1.0             # two constant (e.g. all-zero) maps are identical
    off = ~torch.eye(5, dtype=torch.bool, device=A.device)
    acc = (m(Xte).argmax(1) == yte).float().mean().item()
    return dict(w_sim=wsim[off].mean().item(), f_sim=fsim[off].abs().mean().item(),
                dead=int(dead.sum()), acc=100 * acc, fsim_mat=fsim.cpu().numpy())

def train(m):
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    hist = [dict(epoch=0, **{k: v for k, v in diagnostics(m).items() if k != 'fsim_mat'})]
    for ep in range(1, EPOCHS + 1):
        m.train()
        perm = torch.randperm(len(Xtr), device=dev)
        for i in range(0, len(Xtr), BS):
            idx = perm[i:i + BS]
            loss = F.cross_entropy(m(Xtr[idx]), ytr[idx])
            opt.zero_grad(); loss.backward(); opt.step()
        d = diagnostics(m)
        hist.append(dict(epoch=ep, **{k: v for k, v in d.items() if k != 'fsim_mat'}))
    return pd.DataFrame(hist), d

RUNS = ['① Constant init (bug)', '② He init (fix)', '③ Dead ReLU (bug)', '④ BatchNorm (fix)', '⑤ Leaky ReLU (fix)']
results, models = {}, {}
t0 = time.time()
for run in RUNS:
    models[run] = make_run(run)
    results[run] = train(models[run])
    h = results[run][0].iloc[-1]
    print(f'{run:24s} epoch {EPOCHS}: filter-cos {h.w_sim:.3f} | feature-map corr {h.f_sim:.3f} | '
          f'dead {int(h.dead)}/5 | test acc {h.acc:.1f}%')
print(f'total time {time.time() - t0:.0f}s')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
for run in RUNS:
    h = results[run][0]
    ls = '--' if 'bug' in run else '-'
    axes[0].plot(h.epoch, h.f_sim, ls, marker='o', ms=3, label=run)
    axes[1].plot(h.epoch, h.dead, ls, marker='o', ms=3, label=run)
    axes[2].plot(h.epoch, h.acc, ls, marker='o', ms=3, label=run)
axes[0].set_title('Layer-1 feature-map similarity (1 = identical maps)'); axes[0].set_ylim(0, 1.05)
axes[1].set_title('Dead Layer-1 filters (out of 5)'); axes[1].set_ylim(-0.2, 5.2)
axes[2].set_title(f'Test accuracy % ({dname})')
for a in axes: a.set_xlabel('epoch'); a.grid(alpha=0.3)
axes[2].legend(fontsize=8, loc='best')
plt.tight_layout(); savefig(fig, '10_q5_training_curves'); plt.show()

# Learned filters + feature maps of one test image + similarity matrices
img_t = Xte[3:4]
fig, axes = plt.subplots(len(RUNS), 11, figsize=(20, 2.0 * len(RUNS)))
for r, run in enumerate(RUNS):
    m = models[run]; m.eval()
    with torch.no_grad(): _, a1 = m(img_t, return_a1=True)
    W = m.conv1.weight.detach().cpu().numpy()
    for f in range(5):
        w = W[f].transpose(1, 2, 0); w = (w - w.min()) / (w.max() - w.min() + 1e-12)
        axes[r, f].imshow(w, interpolation='nearest'); axes[r, f].set_title(f'filter {f}', fontsize=7)
        axes[r, 5 + f].imshow(a1[0, f].cpu().numpy(), cmap='gray'); axes[r, 5 + f].set_title(f'map {f}', fontsize=7)
    axes[r, 10].imshow(results[run][1]['fsim_mat'], cmap='viridis', vmin=-1, vmax=1)
    axes[r, 10].set_title('map corr.', fontsize=7)
    axes[r, 0].set_ylabel(run, fontsize=8, rotation=0, ha='right', va='center')
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); savefig(fig, '11_q5_filters_and_maps'); plt.show()

summary = pd.DataFrame({run: results[run][0].iloc[-1][['w_sim', 'f_sim', 'dead', 'acc']] for run in RUNS}).T
summary.columns = ['filter cosine sim', 'feature-map corr', 'dead filters /5', 'test acc %']
display(summary.round(3))

### ✍️ Answer — Q5

Identical feature maps after 20 epochs mean the filters were **never pushed apart**. Gradient descent can only separate filters if they receive *different* gradients. Two independent root causes both remove that difference, and the experiment above reproduces each one:

**Root cause 1: symmetric (constant / zero) weight initialisation → symmetry is never broken.**
If all five filters start with the same weights, they compute the same feature map. The next layer then sees five identical input channels and sends *exactly the same gradient* back to each filter. After every update they are still identical, so this holds by induction for all epochs: ∂L/∂W₁ = ∂L/∂W₂ = … ⇒ W₁(t) = W₂(t) for all t. Adam and SGD both preserve this, because their updates are element-wise functions of identical gradients.
*Evidence (run ①):* filter cosine similarity = **1.000** and feature-map correlation = **1.000** at epoch 0 **and** still at epoch 20 (on GPU, rounding noise may show it as 0.999…). Five filters behave as one.
**Fix:** **random initialisation scaled for ReLU, He/Kaiming init**, W ~ N(0, 2/fan_in) (`nn.init.kaiming_normal_`; Keras `he_normal`). Randomness breaks the symmetry, and the 2/fan_in variance keeps activations from shrinking or exploding. *Run ②:* the filters become distinct (low cosine similarity, see table) and accuracy rises.

**Root cause 2: dead ReLUs (bad bias, too high a learning rate, or a large negative update) → all maps are zero and receive zero gradient.**
If a filter's pre-activation w·x + b is negative for every input, ReLU outputs 0 everywhere and its gradient is 0 everywhere, so the filter can never recover. When every filter dies, all five maps are the *same all-zero map*: "identical feature maps" that training cannot fix. In practice this happens after a too-large learning-rate step pushes biases strongly negative.
*Evidence (run ③):* **5/5 dead filters** for all 20 epochs, feature-map similarity = 1.000, test accuracy at chance (~10%). The weights did not move, because no gradient reached them.
**Fixes:** (a) **BatchNorm before ReLU** (run ④). BN re-centres every pre-activation to zero mean / unit variance per channel, so the bad bias is cancelled and about half the units are active from the first training step: 0 dead filters. (The epoch-0 point is measured in eval mode with BN's *initial* running statistics, so it can still show dead filters before any training step has updated them.) (b) **Leaky ReLU** (run ⑤), whose 0.1 slope lets gradient flow through negative inputs, so the filters keep learning and become distinct. (c) Prevention: **lower the learning rate or use warm-up** with gradient clipping, and initialise biases to 0.

**Other causes worth mentioning in the viva:** too strong L2 weight decay (shrinks all filters toward the same zero vector); a learning rate so *small* that filters barely move from a poor start; having too many filters for an easy task (several learn the same edge). A cheap **monitor**: log the pairwise correlation of Layer-1 maps every epoch, as done here, and raise an alert if it goes above about 0.95.

---
# Viva cheat-sheet (one-line defences)

| Likely question | Answer |
|---|---|
| Is a CNN "convolution" really convolution? | No, it is cross-correlation (no kernel flip). For learned kernels it makes no difference; for hand-made Sobel/emboss it decides the sign of the output. |
| Why does Sobel-X find *vertical* edges? | It is the derivative along x; intensity changes along x happen on vertical boundaries. |
| Why did the edge rows lose ~half their pixels but sharpen/blur didn't? | Edge kernels sum to 0 (zero-mean, signed output); the others sum to 1 (output ≈ local brightness ≥ 0). |
| Why diverging colormap for Stage A only? | Stage A is the only signed stage; sign is what ReLU acts on. B–D are ≥ 0, so gray on [0, m] with the row's shared m. |
| Why reflect padding? | Zero padding creates an artificial edge at the image border. |
| Why don't pooling/GAP layers have parameters? | They compute a fixed max/mean, so there are no weights. |
| Why is the Dense layer 110 in both models? | GAP always outputs 10 numbers (one per channel), independent of kernel size or spatial size. |
| Does padding change parameter count? | No, only output shapes. Parameters depend on K, C_in, C_out only. |
| RF of Model A / B? | 10 / 16 pixels (formula and gradient measurement agree). |
| Is "two 3×3 cheaper than 5×5" always true? | Only if the middle layer isn't wider than the input. In Model B with C_mid = C_out it costs +100 params. |
| Does Conv→Pool→ReLU change gradients? | No for MaxPool: the gradient flows to the same arg-max element and is masked by the same sign. |
| Which pooling for dark text on white? | AvgPool of the two (MaxPool erases dark strokes). Best: invert then MaxPool, or MinPool. |
| How do you detect filter collapse during training? | Track pairwise cosine/correlation of filters and maps and the count of dead units every epoch. |

In [ ]:
# Package every figure for the report (Colab: downloads a zip)
import shutil
shutil.make_archive('cnn_lab_figures', 'zip', FIG_DIR)
print(sorted(os.listdir(FIG_DIR)))
try:
    from google.colab import files
    files.download('cnn_lab_figures.zip')
except Exception:
    print('Not on Colab: figures are in ./figures and cnn_lab_figures.zip')